In [69]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
%pip install pyspark

In [70]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

# Create and config Spark
config = SparkConf().setAppName("bai06").setMaster("local[*]")
sparkContext = SparkContext.getOrCreate(conf=config)
sparkSession = SparkSession.builder.appName("bai06").getOrCreate()

In [71]:
data_path = "/content/drive/MyDrive/data/"

# Read file
movies_data = sparkContext.textFile(data_path + "movies.txt")

ratings1_data = sparkContext.textFile(data_path + "ratings_1.txt")
ratings2_data = sparkContext.textFile(data_path + "ratings_2.txt")

users_data = sparkContext.textFile(data_path + "users.txt")
occupation_data = sparkContext.textFile(data_path + "occupation.txt")

In [72]:
from datetime import datetime
def splition_rating_with_year(line):
    parts = line.strip().split(',')

    if len(parts) < 4:
        return (None, None)

    try:
        user_id = int(parts[0])
        movie_id = int(parts[1])
        rating = float(parts[2])
        timestamp = int(parts[3])
        year = datetime.fromtimestamp(timestamp).year
    except Exception as e:
        year = 2025

    return (year, rating)

# Combine 2 file
ratings1_map = ratings1_data.map(splition_rating_with_year)
ratings2_map = ratings2_data.map(splition_rating_with_year)

# Gộp 2 RDD ratings lại
total_ratings = ratings1_map.union(ratings2_map)


In [73]:

# Create dataset: (year, rating) -> (year, (rating, 1))
year_count = total_ratings.map(lambda x: (x[0], (x[1], 1)))

# Reduce key
# (year, (sum_ratings, total_count))
year_analysis = year_count.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

def calculate_year_average(record):
    year, (sum_ratings, count) = record
    average_rating = sum_ratings / count
    return (year, (count, average_rating))

year_results = year_analysis.map(calculate_year_average).sortByKey()


# Hiển thị tất cả các năm theo định dạng yêu cầu
collection_result_years = year_results.collect()

for year, (ratings, avg) in collection_result_years:
    print(f"{year} - TotalRatings: {ratings}, AverageRating: {avg:.2f}")

2020 - TotalRatings: 184, AverageRating: 3.75
